In [1]:
from pathlib import Path
import numpy as np
import io, time, threading, queue

import imageio.v2 as imageio
from tqdm.auto import tqdm
from astropy.io import fits
from astroquery.mast import Observations

from PIL import Image, ImageDraw
import ipywidgets as widgets
from IPython.display import display


In [2]:
# ---------------------------
# WHAT YOU WANT TO SEE
# ---------------------------
TARGET_NAME = "Mars"
INSTRUMENT  = "NIRCAM"
RADIUS      = "0.3 deg"

# Webb's first Mars NIRCam images are often under GTO Program 1415.
# Set to None to search all programs instead.
PROGRAM_ID = 1415  # or None

# Prefer products that contain frame sequences first
PREFER_KEYS = ("CALINTS", "RATEINTS", "CAL", "RATE", "I2D")

# Limits
MAX_OBS_TO_TRY   = 30
MAX_PRODUCTS_TO_TRY_PER_OBS = 6
MAX_FRAMES_TOTAL = 400

# Video / preview
DO_ALIGN = False          # FFT phase-correlation; set False if too slow
FPS_VIDEO   = 12
FPS_PREVIEW = 12
PREVIEW_BUFFER = 180     # loop last N rendered frames while still working

OUT_DIR = Path("jwst_mars_video")
DATA_DIR = OUT_DIR / "data"
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

video_path = OUT_DIR / f"MARS_{INSTRUMENT}_preview.mp4"
print("Will write:", video_path.resolve())


Will write: /home/jatin/Downloads/jwst_mars_video/MARS_NIRCAM_preview.mp4


In [3]:
from astropy.table import Table, vstack
import warnings

def to_frame_cube(arr: np.ndarray) -> np.ndarray:
    a = np.asarray(arr)
    if a.ndim == 2:
        return a[None, :, :]
    if a.ndim == 3:
        # Heuristic: if the first axis is frame-like, keep it; otherwise move the last axis.
        if a.shape[0] <= max(a.shape[1], a.shape[2]):
            return a
        if a.shape[2] <= max(a.shape[0], a.shape[1]):
            return np.moveaxis(a, 2, 0)
        raise ValueError(f"Unclear 3D layout: {a.shape}")
    if a.ndim == 4:
        # Common JWST case: (nints, ngroups, y, x) -> average groups
        return np.nanmean(a, axis=1)
    raise ValueError(f"Unsupported shape: {a.shape}")

def subgroup_col(tbl):
    for c in tbl.colnames:
        if c.lower() in ("productsubgroupdescription", "productsubgroupdesc"):
            return c
    return None

def normalize_target_str(s: str) -> str:
    return (s or "").strip().upper()

def filter_to_target(obs_tbl, target_name: str):
    tgt = normalize_target_str(target_name)
    for col in ("target_name", "target", "targname", "objname"):
        if col in obs_tbl.colnames:
            vals = np.char.upper(obs_tbl[col].astype(str))
            m = np.char.find(vals, tgt) >= 0
            if np.any(m):
                return obs_tbl[m]
    return obs_tbl

def get_obs_table(target, instrument, radius, program_id=None):
    if program_id is None:
        obs = Observations.query_object(target, radius=radius)
    else:
        obs = Observations.query_criteria(obs_collection="JWST", proposal_id=program_id)
        obs = filter_to_target(obs, target)

    obs = obs[obs["obs_collection"] == "JWST"]
    inst = np.char.upper(obs["instrument_name"].astype(str))
    obs = obs[np.char.find(inst, instrument.upper()) >= 0]

    # Prefer longer/denser observations when available.
    if "t_exptime" in obs.colnames:
        obs.sort("t_exptime")
        obs = obs[::-1]
    elif "t_min" in obs.colnames:
        obs.sort("t_min")
        obs = obs[::-1]
    return obs

def pick_products_for_obs(obs_row, prefer_keys=PREFER_KEYS):
    products = Observations.get_product_list(obs_row)
    p = Observations.filter_products(products, productType="SCIENCE", extension="fits")
    if len(p) == 0:
        return None

    fn = np.char.lower(p["productFilename"].astype(str))
    sc = subgroup_col(p)

    # Strongly prefer true integration cubes first.
    selected_chunks = []
    used = np.zeros(len(p), dtype=bool)

    for key in prefer_keys:
        key_l = key.lower()
        mask = np.zeros(len(p), dtype=bool)
        if sc is not None:
            mask |= (np.char.lower(p[sc].astype(str)) == key_l)
        mask |= (np.char.find(fn, key_l) >= 0)
        take = mask & ~used
        if np.any(take):
            selected_chunks.append(p[take])
            used |= take

    # Then allow a few other image-like FITS products as fallback.
    extra_mask = (
        (np.char.find(fn, "trapsfilled") >= 0) |
        (np.char.find(fn, "_cal") >= 0) |
        (np.char.find(fn, "_rate") >= 0)
    ) & ~used
    if np.any(extra_mask):
        selected_chunks.append(p[extra_mask])
        used |= extra_mask

    if np.any(~used):
        selected_chunks.append(p[~used])

    if not selected_chunks:
        return p
    return vstack(selected_chunks)

def _row_to_one_row_table(product_row):
    return Table({name: [product_row[name]] for name in product_row.colnames})

def download_one(product_row, download_dir):
    one = _row_to_one_row_table(product_row)
    manifest = Observations.download_products(
        one,
        download_dir=str(download_dir),
        cache=True,
    )
    if len(manifest) == 0:
        raise RuntimeError("MAST returned an empty download manifest.")
    local_col = "Local Path" if "Local Path" in manifest.colnames else "local_path"
    return Path(manifest[local_col][0])

def read_frames(fits_path: Path):
    with fits.open(fits_path, memmap=False) as hdul:
        names = [hdu.name for hdu in hdul]
        if "SCI" not in hdul:
            raise RuntimeError(f"No SCI extension. HDUs={names}")
        data = hdul["SCI"].data
        if data is None:
            raise RuntimeError("SCI extension exists but contains no data.")
        cube = to_frame_cube(data).astype(np.float32)

    if cube.ndim != 3:
        raise RuntimeError(f"Expected a 3D frame cube, got shape {cube.shape}")
    if cube.shape[0] < 1:
        raise RuntimeError(f"Cube contains no frames: {cube.shape}")
    return cube

def finite_fill_value(arr, default=0.0):
    a = np.asarray(arr)
    finite = a[np.isfinite(a)]
    if finite.size == 0:
        return float(default)
    return float(np.median(finite))

def background_subtract(frame):
    fill = finite_fill_value(frame, default=0.0)
    if not np.any(np.isfinite(frame)):
        return np.zeros_like(frame, dtype=np.float32)
    return np.nan_to_num(frame - fill, nan=0.0).astype(np.float32)

def find_disk_bbox(frame, pad=30):
    fill = finite_fill_value(frame, default=0.0)
    m = np.nan_to_num(frame, nan=fill)
    if m.size == 0:
        return 0, 0, 0, 0
    if not np.any(np.isfinite(m)):
        h, w = m.shape
        return 0, h, 0, w
    thr = np.nanpercentile(m, 90)
    mask = m > thr
    if not np.any(mask):
        h, w = m.shape
        return 0, h, 0, w
    ys, xs = np.where(mask)
    y0, y1 = ys.min(), ys.max()
    x0, x1 = xs.min(), xs.max()
    h, w = m.shape
    y0 = max(0, y0 - pad); y1 = min(h, y1 + pad)
    x0 = max(0, x0 - pad); x1 = min(w, x1 + pad)
    return y0, y1, x0, x1

def robust_vmin_vmax(frames3d):
    a = np.asarray(frames3d)
    finite = a[np.isfinite(a)]
    if finite.size == 0:
        return 0.0, 1.0
    vmin, vmax = np.percentile(finite, [2, 99.7])
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
        vmin, vmax = float(np.min(finite)), float(np.max(finite))
    if vmax <= vmin:
        vmax = vmin + 1.0
    return float(vmin), float(vmax)

def scale_to_uint8(img, vmin, vmax):
    x = np.nan_to_num(img, nan=vmin)
    x = np.clip((x - vmin) / (vmax - vmin + 1e-12), 0, 1)
    x = np.arcsinh(10 * x) / np.arcsinh(10)
    return (255 * x).astype(np.uint8)

def annotate(rgb, text):
    im = Image.fromarray(rgb)
    draw = ImageDraw.Draw(im)
    draw.rectangle([0, 0, im.size[0], 26], fill=(0, 0, 0))
    draw.text((6, 5), text, fill=(255, 255, 255))
    return np.array(im)

def phase_corr_shift(ref, img):
    ref = np.nan_to_num(ref, nan=finite_fill_value(ref, default=0.0))
    img = np.nan_to_num(img, nan=finite_fill_value(img, default=0.0))
    if not np.any(ref) and not np.any(img):
        return 0, 0

    F = np.fft.fft2(ref)
    G = np.fft.fft2(img)
    R = F * np.conj(G)
    denom = np.abs(R)
    good = denom > 1e-9
    R = np.where(good, R / denom, 0.0)
    corr = np.abs(np.fft.ifft2(R))
    y, x = np.unravel_index(np.argmax(corr), corr.shape)
    if y > corr.shape[0] // 2:
        y -= corr.shape[0]
    if x > corr.shape[1] // 2:
        x -= corr.shape[1]
    return int(y), int(x)

def align_to_reference(frames3d):
    ref = np.nan_to_num(frames3d[0], nan=finite_fill_value(frames3d[0], default=0.0))
    out = [ref.astype(np.float32)]
    for i in range(1, len(frames3d)):
        cur = np.nan_to_num(frames3d[i], nan=finite_fill_value(frames3d[i], default=0.0))
        dy, dx = phase_corr_shift(ref, cur)
        out.append(np.roll(cur, shift=(dy, dx), axis=(0, 1)).astype(np.float32))
    return np.stack(out, axis=0)

def rgb_png_bytes(rgb):
    buf = io.BytesIO()
    Image.fromarray(rgb).save(buf, format="PNG")
    return buf.getvalue()

def ensure_preview_started():
    global img_widget, frame_q, stop_flag, _preview_thread

    if "img_widget" not in globals():
        img_widget = widgets.Image(format="png")
        display(img_widget)

    if "frame_q" not in globals() or not isinstance(frame_q, queue.Queue):
        frame_q = queue.Queue(maxsize=PREVIEW_BUFFER)

    if "stop_flag" not in globals() or not isinstance(stop_flag, dict):
        stop_flag = {"stop": False}
    else:
        stop_flag["stop"] = False

    def preview_worker():
        last = []
        i = 0
        while not stop_flag["stop"]:
            try:
                b = frame_q.get(timeout=0.2)
                last.append(b)
                if len(last) > PREVIEW_BUFFER:
                    last = last[-PREVIEW_BUFFER:]
                img_widget.value = b
            except queue.Empty:
                if last:
                    img_widget.value = last[i % len(last)]
                    i += 1
                    time.sleep(1 / FPS_PREVIEW)

    if "_preview_thread" not in globals() or not getattr(_preview_thread, "is_alive", lambda: False)():
        _preview_thread = threading.Thread(target=preview_worker, daemon=True)
        _preview_thread.start()


In [4]:
ensure_preview_started()


Image(value=b'')

In [4]:
obs = get_obs_table(TARGET_NAME, INSTRUMENT, RADIUS, PROGRAM_ID)
print(f"Found {len(obs)} JWST obs rows matching {TARGET_NAME}/{INSTRUMENT}.")

best = None  # (nframes, fits_path, frames3d)

for i_obs, obs_row in enumerate(obs[:MAX_OBS_TO_TRY], start=1):
    print(f"\n=== Observation {i_obs}/{min(len(obs), MAX_OBS_TO_TRY)} ===")
    prods = pick_products_for_obs(obs_row)
    if prods is None or len(prods) == 0:
        print("No candidate FITS science products.")
        continue

    # Try larger files first when size exists.
    if "size" in prods.colnames:
        prods.sort("size")
        prods = prods[::-1]

    for j, pr in enumerate(prods[:MAX_PRODUCTS_TO_TRY_PER_OBS], start=1):
        filename = pr["productFilename"] if "productFilename" in pr.colnames else "<unknown>"
        size = pr["size"] if "size" in pr.colnames else "?"
        print(f"  Trying product {j}: {filename}  size={size}")
        try:
            fp = download_one(pr, DATA_DIR)
            print(f"    Downloaded/cached: {fp}")

            fr = read_frames(fp)
            print(f"    Read cube shape: {fr.shape}")

            n = fr.shape[0]
            if best is None or n > best[0]:
                best = (n, fp, fr)
                print(f"    New best candidate with {n} frame(s).")

            if n >= 20:
                print("    Found movie-like cube; stopping early.")
                break
        except Exception as e:
            print(f"    FAILED: {type(e).__name__}: {e}")
            continue

    if best is not None and best[0] >= 20:
        break

if best is None:
    stop_flag["stop"] = True
    raise RuntimeError("Could not find a usable Mars NIRCam FITS cube (SCI) from the selected search.")

nframes, fits_path, frames = best
print(f"\nUsing: {fits_path.name} with {nframes} frame(s).")

# Limit frames for memory/time.
if nframes > MAX_FRAMES_TOTAL:
    idx = np.linspace(0, nframes - 1, MAX_FRAMES_TOTAL).round().astype(int)
    frames = frames[idx]
    nframes = len(frames)
    print("Downsampled to", nframes, "frames.")


Found 2 JWST obs rows matching Mars/NIRCAM.

=== Observation 1/2 ===
  Trying product 1: jw01415001001_03102_00001_nrcb1_rateints.fits  size=5195520
INFO: Found cached file jwst_mars_video/data/mastDownload/JWST/jw01415001001_03102_00001_nrcb1/jw01415001001_03102_00001_nrcb1_rateints.fits with expected size 5195520. [astroquery.query]
    Downloaded/cached: jwst_mars_video/data/mastDownload/JWST/jw01415001001_03102_00001_nrcb1/jw01415001001_03102_00001_nrcb1_rateints.fits
    Read cube shape: (10, 160, 160)
    New best candidate with 10 frame(s).
  Trying product 2: jw01415001001_03102_00003_nrcb1_rateints.fits  size=5195520
INFO: Found cached file jwst_mars_video/data/mastDownload/JWST/jw01415001001_03102_00003_nrcb1/jw01415001001_03102_00003_nrcb1_rateints.fits with expected size 5195520. [astroquery.query]
    Downloaded/cached: jwst_mars_video/data/mastDownload/JWST/jw01415001001_03102_00003_nrcb1/jw01415001001_03102_00003_nrcb1_rateints.fits
    Read cube shape: (10, 160, 160)
  

In [5]:
ensure_preview_started()

# preprocess: background-subtract + crop around disk using median frame
if not np.any(np.isfinite(frames)):
    raise RuntimeError("Selected FITS cube contains no finite pixel values.")

with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    med = np.nanmedian(frames, axis=0)

if not np.any(np.isfinite(med)):
    fill = finite_fill_value(frames, default=0.0)
    med = np.nan_to_num(frames[0], nan=fill)

y0, y1, x0, x1 = find_disk_bbox(med, pad=40)

frames_bs = np.array(
    [background_subtract(f)[y0:y1, x0:x1] for f in frames],
    dtype=np.float32
)

if frames_bs.size == 0:
    raise RuntimeError("Cropping produced an empty frame cube; try increasing the pad or disabling cropping.")
if not np.any(np.isfinite(frames_bs)):
    raise RuntimeError("Preprocessed frame cube contains no finite values.")

# optional alignment on the cropped frames
if DO_ALIGN and len(frames_bs) > 1:
    frames_bs = align_to_reference(frames_bs)

vmin, vmax = robust_vmin_vmax(frames_bs)

with imageio.get_writer(video_path, fps=FPS_VIDEO) as writer:
    for i in tqdm(range(nframes), desc="Rendering"):
        g = scale_to_uint8(frames_bs[i], vmin, vmax)
        rgb = np.stack([g, g, g], axis=-1)

        label = f"Mars | {INSTRUMENT} | {i+1}/{nframes} | {fits_path.name}"
        rgb = annotate(rgb, label)

        # push to live preview (non-blocking)
        try:
            frame_q.put_nowait(rgb_png_bytes(rgb))
        except queue.Full:
            pass

        writer.append_data(rgb)

stop_flag["stop"] = True
print("Done! Video written to:", video_path.resolve())


Image(value=b'')

Rendering:   0%|          | 0/10 [00:00<?, ?it/s]

Done! Video written to: /home/jatin/Downloads/jwst_mars_video/MARS_NIRCAM_preview.mp4
